In [ ]:
# ============================================
# BLOOD CELL CLASSIFICATION (Colab) - kagglehub + Pretrained CNN (PyTorch ResNet50)
# Dataset: https://www.kaggle.com/datasets/unclesamulus/blood-cells-image-dataset
# ============================================

# 1) Install deps (run once)
!pip -q install kagglehub torch torchvision scikit-learn

# 2) Download dataset via kagglehub
import os
import kagglehub

path = kagglehub.dataset_download("unclesamulus/blood-cells-image-dataset")
print("Path to dataset files:", path)

# 3) Find the images folder automatically (robust)
def find_images_dir(base_path: str) -> str:
    """
    Finds a directory that looks like an ImageFolder root:
    it contains subfolders and those subfolders contain image files.
    """
    image_exts = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp")
    candidates = []
    for root, dirs, files in os.walk(base_path):
        if not dirs:
            continue
        # check if any immediate subdir contains at least 1 image
        ok_subdirs = 0
        for d in dirs[:10]:
            sub = os.path.join(root, d)
            try:
                for f in os.listdir(sub):
                    if f.lower().endswith(image_exts):
                        ok_subdirs += 1
                        break
            except Exception:
                pass
        if ok_subdirs >= 2:  # looks like class folders
            candidates.append(root)

    if not candidates:
        raise FileNotFoundError(
            "Could not auto-detect the images folder. "
            "Please print os.listdir(path) and share the output."
        )

    # Prefer paths that include 'images' in their name, else choose shortest
    candidates.sort(key=lambda p: (("images" not in p.lower()), len(p)))
    return candidates[0]

DATA_DIR = find_images_dir(path)
print("Detected DATA_DIR:", DATA_DIR)
print("Class folders:", os.listdir(DATA_DIR))

# 4) Imports
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import classification_report, confusion_matrix

# 5) Config
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_FROZEN = 8        # stage 1
EPOCHS_FINETUNE = 5      # stage 2
SEED = 42
VAL_SPLIT = 0.2
LR_FROZEN = 1e-3
LR_FINETUNE = 1e-5
NUM_WORKERS = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.manual_seed(SEED)

# 6) Transforms
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.ColorJitter(brightness=0.08, contrast=0.12, saturation=0.08),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# 7) Dataset + split
full_ds = ImageFolder(DATA_DIR, transform=train_tfms)
class_names = full_ds.classes
num_classes = len(class_names)
print("Classes:", class_names)
print("Num classes:", num_classes)
print("Total images:", len(full_ds))

val_size = int(VAL_SPLIT * len(full_ds))
train_size = len(full_ds) - val_size
train_ds, val_ds = random_split(full_ds, [train_size, val_size])

# Make validation deterministic transform
val_ds.dataset.transform = val_tfms

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

# 8) Model: Pretrained ResNet50
weights = torchvision.models.ResNet50_Weights.DEFAULT
model = torchvision.models.resnet50(weights=weights)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# 9) Helpers
def accuracy_from_logits(logits, y):
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_acc, n = 0.0, 0.0, 0

    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = criterion(logits, y)

            if train:
                loss.backward()
                optimizer.step()

        bs = x.size(0)
        total_loss += loss.item() * bs
        total_acc += accuracy_from_logits(logits, y) * bs
        n += bs

    return total_loss / n, total_acc / n

@torch.no_grad()
def evaluate_and_report():
    model.eval()
    y_true, y_pred = [], []
    for x, y in val_loader:
        x = x.to(device, non_blocking=True)
        logits = model(x)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(y.numpy())

    print("\nClassification Report:\n")
    print(classification_report(y_true, y_pred, target_names=class_names))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

# 10) Stage 1: Freeze backbone, train classifier head
for name, param in model.named_parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=LR_FROZEN)

print("\n=== STAGE 1: Training classifier head (backbone frozen) ===")
best_val_acc = 0.0
best_path = "best_resnet50_bloodcells.pth"

for epoch in range(EPOCHS_FROZEN):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch+1}/{EPOCHS_FROZEN} | "
          f"train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | "
          f"val_loss={va_loss:.4f} val_acc={va_acc:.4f}")

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "classes": class_names,
            "data_dir": DATA_DIR,
        }, best_path)

print("Best val acc (stage 1):", best_val_acc)

# 11) Stage 2: Fine-tune last block (unfreeze some layers)
# Reload best stage-1 weights first
ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])

# Unfreeze layer4 + fc (common fine-tuning strategy for ResNet)
for name, param in model.named_parameters():
    param.requires_grad = False
for name, param in model.named_parameters():
    if name.startswith("layer4") or name.startswith("fc"):
        param.requires_grad = True

# Optimizer on trainable params only
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(trainable_params, lr=LR_FINETUNE)

print("\n=== STAGE 2: Fine-tuning (layer4 + fc unfrozen) ===")
best_val_acc_ft = best_val_acc

for epoch in range(EPOCHS_FINETUNE):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch+1}/{EPOCHS_FINETUNE} | "
          f"train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | "
          f"val_loss={va_loss:.4f} val_acc={va_acc:.4f}")

    if va_acc > best_val_acc_ft:
        best_val_acc_ft = va_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "classes": class_names,
            "data_dir": DATA_DIR,
        }, best_path)

print("Best val acc (after fine-tune):", best_val_acc_ft)

# 12) Final evaluation report
model.load_state_dict(torch.load(best_path, map_location=device)["model_state_dict"])
evaluate_and_report()

# 13) Save final checkpoint (same as best_path) and offer download in Colab
from google.colab import files
files.download(best_path)

print("\nDONE ✅ Model saved as:", best_path)


100%|██████████| 268M/268M [00:13<00:00, 21.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/unclesamulus/blood-cells-image-dataset/versions/2
Detected DATA_DIR: /root/.cache/kagglehub/datasets/unclesamulus/blood-cells-image-dataset/versions/2/bloodcells_dataset
Class folders: ['ig', 'lymphocyte', 'platelet', 'monocyte', 'eosinophil', 'erythroblast', 'neutrophil', 'basophil']
Using device: cuda
Classes: ['basophil', 'eosinophil', 'erythroblast', 'ig', 'lymphocyte', 'monocyte', 'neutrophil', 'platelet']
Num classes: 8
Total images: 17092
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 203MB/s]



=== STAGE 1: Training classifier head (backbone frozen) ===
Epoch 1/8 | train_loss=0.9260 train_acc=0.7631 | val_loss=0.5429 val_acc=0.8686
Epoch 2/8 | train_loss=0.4735 train_acc=0.8786 | val_loss=0.3864 val_acc=0.8991
Epoch 3/8 | train_loss=0.3676 train_acc=0.9032 | val_loss=0.3250 val_acc=0.9131
Epoch 4/8 | train_loss=0.3145 train_acc=0.9171 | val_loss=0.2827 val_acc=0.9225
Epoch 5/8 | train_loss=0.2770 train_acc=0.9262 | val_loss=0.2691 val_acc=0.9233
Epoch 6/8 | train_loss=0.2523 train_acc=0.9317 | val_loss=0.2500 val_acc=0.9295
Epoch 7/8 | train_loss=0.2325 train_acc=0.9353 | val_loss=0.2466 val_acc=0.9298
Epoch 8/8 | train_loss=0.2179 train_acc=0.9388 | val_loss=0.2432 val_acc=0.9239
Best val acc (stage 1): 0.929783499436186

=== STAGE 2: Fine-tuning (layer4 + fc unfrozen) ===
Epoch 1/5 | train_loss=0.1585 train_acc=0.9519 | val_loss=0.1582 val_acc=0.9508
Epoch 2/5 | train_loss=0.0877 train_acc=0.9737 | val_loss=0.1368 val_acc=0.9576
Epoch 3/5 | train_loss=0.0545 train_acc=0.98

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


DONE ✅ Model saved as: best_resnet50_bloodcells.pth
